In [1]:
# ============================================================
# CELL 1 — LOAD AND AUDIT RAW DATA
# ============================================================

import pandas as pd
import numpy as np

RAW_PATH = "../data/raw/TEJAS_RAW_DATA.csv"
ROUTE_PATH = "../data/raw/ROUTE.csv"

df = pd.read_csv(RAW_PATH)
route_df = pd.read_csv(ROUTE_PATH)

print("RAW DATA SHAPE:", df.shape)
print("ROUTE DATA SHAPE:", route_df.shape)

print("\nRAW DATA COLUMNS:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

print("\nROUTE DATA COLUMNS:")
for i, col in enumerate(route_df.columns, 1):
    print(f"{i:02d}. {col}")

print("\nTARGET SUMMARY:")
print(df["EV Suitability Score"].describe())

print("\nTERRAIN DISTRIBUTION:")
print(route_df["Terrain_Class"].value_counts(dropna=False))

print("\nMISSING VALUES IN IMPORTANT COLUMNS:")
important_cols = [
    "Depot ID",
    "Depot Name",
    "District",
    "Month",
    "Buses Allocated",
    "Schedules Allocated",
    "Effective KM",
    "Passengers",
    "Estimated Diesel Litres",
    "Estimated CO2 (Tonnes)",
    "Diesel Cost (INR/KM)",
    "EV Cost (INR/KM)",
    "Estimated EV Energy (MWh)",
    "Potential EV OPEX Saving (INR)",
    "EV Battery Capacity (kWh)",
    "EV Range (KM)",
    "Charging Power (kW)",
    "Electricity Tariff (INR/kWh)",
    "EV Suitability Score"
]

print(df[important_cols].isna().sum())

RAW DATA SHAPE: (5520, 20)
ROUTE DATA SHAPE: (92, 9)

RAW DATA COLUMNS:
01. Depot ID
02. Depot Name
03. District
04. Month
05. Buses Allocated
06. Schedules Allocated
07. Effective KM
08. Passengers
09. Estimated Diesel Litres
10. Estimated CO2 (Tonnes)
11. Diesel Cost (INR/KM)
12. EV Cost (INR/KM)
13. Estimated EV Energy (MWh)
14. Potential EV OPEX Saving (INR)
15. EV Battery Capacity (kWh)
16. EV Range (KM)
17. Charging Power (kW)
18. Electricity Tariff (INR/kWh)
19. EV Suitability Score
20. Year

ROUTE DATA COLUMNS:
01. Route_ID
02. Depot_ID
03. Depot_Name
04. District
05. Origin_or_Depot
06. Destination_or_Corridor
07. Distance_KM
08. Terrain_Class
09. Terrain_Score

TARGET SUMMARY:
count    5520.000000
mean        0.463777
std         0.116270
min         0.188500
25%         0.399000
50%         0.443500
75%         0.515200
max         0.776200
Name: EV Suitability Score, dtype: float64

TERRAIN DISTRIBUTION:
Terrain_Class
Flat/Rolling    57
Rolling         18
Flat             7

In [2]:
# ============================================================
# CELL 2 — AUDIT CURRENT EV SUITABILITY SCORE
# ============================================================

# Select numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Correlation with current suitability score
corr = (
    df[numeric_cols]
    .corr()["EV Suitability Score"]
    .sort_values(ascending=False)
)

print("CORRELATION WITH CURRENT EV SUITABILITY SCORE")
print("=" * 55)
print(corr)

# Show target formula-related columns
formula_cols = [
    "Effective KM",
    "Estimated Diesel Litres",
    "Diesel Cost (INR/KM)",
    "EV Cost (INR/KM)",
    "Estimated EV Energy (MWh)",
    "Potential EV OPEX Saving (INR)",
    "EV Battery Capacity (kWh)",
    "EV Range (KM)",
    "Charging Power (kW)",
    "Electricity Tariff (INR/kWh)",
    "EV Suitability Score"
]

print("\n\nCURRENT TARGET + POSSIBLE TARGET COMPONENTS")
print("=" * 55)
print(df[formula_cols].describe().T)

# Check whether the target has the same value pattern within depots
print("\n\nSUITABILITY SCORE BY DEPOT")
print("=" * 55)

depot_summary = (
    df.groupby(["Depot ID", "Depot Name"])["EV Suitability Score"]
      .agg(["count", "mean", "std", "min", "max"])
      .sort_values("mean", ascending=False)
)

print(depot_summary.head(15))

# Number of unique target values
print("\n\nTARGET UNIQUE VALUES:", df["EV Suitability Score"].nunique())

# Check whether score changes over time
print("\n\nYEAR-WISE TARGET")
print("=" * 55)

print(
    df.groupby("Year")["EV Suitability Score"]
      .agg(["count", "mean", "std", "min", "max"])
)

CORRELATION WITH CURRENT EV SUITABILITY SCORE
EV Suitability Score              1.000000
Passengers                        0.682938
Buses Allocated                   0.606243
Schedules Allocated               0.500606
Potential EV OPEX Saving (INR)    0.282849
Estimated CO2 (Tonnes)            0.282849
Effective KM                      0.282849
Estimated Diesel Litres           0.282849
Estimated EV Energy (MWh)         0.282831
Year                             -0.149270
Diesel Cost (INR/KM)                   NaN
EV Cost (INR/KM)                       NaN
EV Battery Capacity (kWh)              NaN
EV Range (KM)                          NaN
Charging Power (kW)                    NaN
Electricity Tariff (INR/kWh)           NaN
Name: EV Suitability Score, dtype: float64


CURRENT TARGET + POSSIBLE TARGET COMPONENTS
                                 count          mean           std  \
Effective KM                    5520.0  3.512494e+05  1.171666e+05   
Estimated Diesel Litres         5520.

In [3]:
# ============================================================
# CELL 3 — CHECK FEATURE VARIATION
# ============================================================

check_cols = [
    "Buses Allocated",
    "Schedules Allocated",
    "Effective KM",
    "Passengers",
    "Estimated Diesel Litres",
    "Estimated CO2 (Tonnes)",
    "Diesel Cost (INR/KM)",
    "EV Cost (INR/KM)",
    "Estimated EV Energy (MWh)",
    "Potential EV OPEX Saving (INR)",
    "EV Battery Capacity (kWh)",
    "EV Range (KM)",
    "Charging Power (kW)",
    "Electricity Tariff (INR/kWh)"
]

print("UNIQUE VALUES / VARIATION")
print("=" * 70)

for col in check_cols:
    unique_count = df[col].nunique()
    min_val = df[col].min()
    max_val = df[col].max()

    print(
        f"{col:<40} "
        f"Unique: {unique_count:<5} "
        f"Min: {min_val:<12.2f} "
        f"Max: {max_val:<12.2f}"
    )

print("\n\nROUTE/TERRAIN DATA")
print("=" * 70)

print("Unique depots in RAW DATA :", df["Depot ID"].nunique())
print("Unique depots in ROUTE   :", route_df["Depot_ID"].nunique())

print("\nTerrain classes:")
print(route_df["Terrain_Class"].value_counts())

print("\nTerrain scores:")
print(
    route_df[["Terrain_Class", "Terrain_Score"]]
    .drop_duplicates()
    .sort_values("Terrain_Score", ascending=False)
)

print("\n\nCURRENT TARGET RANGE BY DEPOT")
print("=" * 70)

depot_target = (
    df.groupby(["Depot ID", "Depot Name"])["EV Suitability Score"]
      .agg(["mean", "min", "max"])
      .sort_values("mean", ascending=False)
)

print(depot_target.head(10))

UNIQUE VALUES / VARIATION
Buses Allocated                          Unique: 78    Min: 27.00        Max: 105.00      
Schedules Allocated                      Unique: 72    Min: 20.00        Max: 91.00       
Effective KM                             Unique: 5376  Min: 116264.00    Max: 821294.00   
Passengers                               Unique: 5386  Min: 111005.00    Max: 1240216.00  
Estimated Diesel Litres                  Unique: 5376  Min: 28492.59     Max: 201272.88   
Estimated CO2 (Tonnes)                   Unique: 5376  Min: 76.36        Max: 539.41      
Diesel Cost (INR/KM)                     Unique: 1     Min: 51.00        Max: 51.00       
EV Cost (INR/KM)                         Unique: 1     Min: 27.00        Max: 27.00       
Estimated EV Energy (MWh)                Unique: 682   Min: 145.00       Max: 1027.00     
Potential EV OPEX Saving (INR)           Unique: 5376  Min: 2790336.00   Max: 19711056.00 
EV Battery Capacity (kWh)                Unique: 1     Min: 250.

In [4]:
# ============================================================
# CELL 4 — DEPOT-LEVEL EV TRANSITION FACTORS
# ============================================================

# Make a copy so the raw dataframe remains untouched
analysis_df = df.copy()

# Convert Month to datetime
analysis_df["Month"] = pd.to_datetime(analysis_df["Month"])

# Create operational indicators
analysis_df["KM_per_Bus"] = (
    analysis_df["Effective KM"] /
    analysis_df["Buses Allocated"]
)

analysis_df["Passengers_per_Bus"] = (
    analysis_df["Passengers"] /
    analysis_df["Buses Allocated"]
)

analysis_df["Passengers_per_KM"] = (
    analysis_df["Passengers"] /
    analysis_df["Effective KM"]
)

analysis_df["Schedules_per_Bus"] = (
    analysis_df["Schedules Allocated"] /
    analysis_df["Buses Allocated"]
)

# Approximate monthly EV energy using the project's
# existing 1.25 kWh/km planning assumption
analysis_df["EV_Energy_MWh_Calculated"] = (
    analysis_df["Effective KM"] * 1.25 / 1000
)

# Approximate daily KM per bus
analysis_df["Days_in_Month"] = analysis_df["Month"].dt.days_in_month

analysis_df["Daily_KM_per_Bus"] = (
    analysis_df["Effective KM"] /
    analysis_df["Days_in_Month"] /
    analysis_df["Buses Allocated"]
)

# Compare daily operating requirement with the assumed
# 200 km EV range
analysis_df["Range_Utilization"] = (
    analysis_df["Daily_KM_per_Bus"] / 200
)

# Depot-level summary
depot_factors = (
    analysis_df
    .groupby(["Depot ID", "Depot Name", "District"])
    .agg(
        Avg_KM=("Effective KM", "mean"),
        Avg_Passengers=("Passengers", "mean"),
        Avg_Buses=("Buses Allocated", "mean"),
        Avg_Schedules=("Schedules Allocated", "mean"),
        Avg_KM_per_Bus=("KM_per_Bus", "mean"),
        Avg_Passengers_per_Bus=("Passengers_per_Bus", "mean"),
        Avg_Passengers_per_KM=("Passengers_per_KM", "mean"),
        Avg_Schedules_per_Bus=("Schedules_per_Bus", "mean"),
        Avg_OPEX_Saving=("Potential EV OPEX Saving (INR)", "mean"),
        Avg_EV_Energy_MWh=("EV_Energy_MWh_Calculated", "mean"),
        Avg_Daily_KM_per_Bus=("Daily_KM_per_Bus", "mean"),
        Avg_Range_Utilization=("Range_Utilization", "mean"),
        Old_Suitability=("EV Suitability Score", "mean")
    )
    .reset_index()
)

# Merge route/terrain information
terrain_cols = [
    "Depot_ID",
    "Terrain_Class",
    "Terrain_Score"
]

depot_factors = depot_factors.merge(
    route_df[terrain_cols],
    left_on="Depot ID",
    right_on="Depot_ID",
    how="left"
)

depot_factors.drop(columns=["Depot_ID"], inplace=True)

print("DEPOT-LEVEL DATASET SHAPE:", depot_factors.shape)

print("\nFIRST 10 DEPOTS")
print("=" * 80)

print(
    depot_factors[
        [
            "Depot ID",
            "Depot Name",
            "Avg_KM_per_Bus",
            "Avg_Passengers_per_Bus",
            "Avg_Daily_KM_per_Bus",
            "Avg_Range_Utilization",
            "Avg_OPEX_Saving",
            "Terrain_Class",
            "Terrain_Score",
            "Old_Suitability"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\n\nSUMMARY OF NEW FACTORS")
print("=" * 80)

print(
    depot_factors[
        [
            "Avg_KM_per_Bus",
            "Avg_Passengers_per_Bus",
            "Avg_Daily_KM_per_Bus",
            "Avg_Range_Utilization",
            "Avg_OPEX_Saving",
            "Terrain_Score"
        ]
    ].describe().T
)

print("\n\nMISSING TERRAIN VALUES:",
      depot_factors["Terrain_Score"].isna().sum())

DEPOT-LEVEL DATASET SHAPE: (92, 18)

FIRST 10 DEPOTS
 Depot ID      Depot Name  Avg_KM_per_Bus  Avg_Passengers_per_Bus  Avg_Daily_KM_per_Bus  Avg_Range_Utilization  Avg_OPEX_Saving Terrain_Class  Terrain_Score  Old_Suitability
KSRTC-001           ADOOR     6968.845160             9350.088077            229.141142               1.145706        4891721.6       Rolling           0.75         0.388118
KSRTC-002       ALAPPUZHA     6879.450124             9986.222693            226.218301               1.131092       10549751.6          Flat           1.00         0.506895
KSRTC-003           ALUVA     6684.816292             9639.047716            219.820903               1.099105        8116700.8  Flat/Rolling           0.90         0.477978
KSRTC-004        ANKAMALY     6676.980791             9629.283388            219.565755               1.097829        8116698.0  Flat/Rolling           0.90         0.477813
KSRTC-005        ATTINGAL     6229.829225            10553.062988            

In [5]:
# ============================================================
# CELL 5 — CREATE EV TRANSITION SUITABILITY COMPONENTS
# ============================================================

target_df = depot_factors.copy()

# ------------------------------------------------------------
# 1. ECONOMIC BENEFIT SCORE
# ------------------------------------------------------------
# Use OPEX saving per bus so large depots are not automatically
# favored simply because they have more buses.

target_df["OPEX_Saving_per_Bus"] = (
    target_df["Avg_OPEX_Saving"] /
    target_df["Avg_Buses"]
)

# Min-Max normalization: 0 = lowest benefit, 1 = highest
target_df["Economic_Benefit_Score"] = (
    (target_df["OPEX_Saving_per_Bus"] -
     target_df["OPEX_Saving_per_Bus"].min())
    /
    (target_df["OPEX_Saving_per_Bus"].max() -
     target_df["OPEX_Saving_per_Bus"].min())
)


# ------------------------------------------------------------
# 2. OPERATIONAL UTILIZATION SCORE
# ------------------------------------------------------------
# Combine:
# - KM operated per bus
# - Passengers carried per bus

km_norm = (
    (target_df["Avg_KM_per_Bus"] -
     target_df["Avg_KM_per_Bus"].min())
    /
    (target_df["Avg_KM_per_Bus"].max() -
     target_df["Avg_KM_per_Bus"].min())
)

passenger_norm = (
    (target_df["Avg_Passengers_per_Bus"] -
     target_df["Avg_Passengers_per_Bus"].min())
    /
    (target_df["Avg_Passengers_per_Bus"].max() -
     target_df["Avg_Passengers_per_Bus"].min())
)

target_df["Operational_Utilization_Score"] = (
    0.60 * km_norm +
    0.40 * passenger_norm
)


# ------------------------------------------------------------
# 3. TERRAIN / ROUTE FEASIBILITY SCORE
# ------------------------------------------------------------
# Terrain_Score comes directly from ROUTE.csv.
#
# Flat          = 1.00
# Flat/Rolling  = 0.90
# Rolling       = 0.75
# Hilly         = 0.40
# Steep         = 0.20

target_df["Route_Feasibility_Score"] = (
    target_df["Terrain_Score"]
)


# ------------------------------------------------------------
# 4. FINAL EV TRANSITION SUITABILITY SCORE
# ------------------------------------------------------------

target_df["EV_Transition_Suitability_Score"] = (
    0.40 * target_df["Economic_Benefit_Score"] +
    0.30 * target_df["Operational_Utilization_Score"] +
    0.30 * target_df["Route_Feasibility_Score"]
)

# Keep score safely between 0 and 1
target_df["EV_Transition_Suitability_Score"] = (
    target_df["EV_Transition_Suitability_Score"]
    .clip(0, 1)
)


# ------------------------------------------------------------
# 5. CREATE DECISION CLASSES
# ------------------------------------------------------------

def classify_transition(score):
    if score >= 0.70:
        return "EV Preferred"
    elif score >= 0.45:
        return "EV Conditional"
    else:
        return "Diesel Preferred"

target_df["EV_Transition_Class"] = (
    target_df["EV_Transition_Suitability_Score"]
    .apply(classify_transition)
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("NEW TARGET CREATED")
print("=" * 80)

print(
    target_df[
        [
            "Depot ID",
            "Depot Name",
            "Economic_Benefit_Score",
            "Operational_Utilization_Score",
            "Route_Feasibility_Score",
            "EV_Transition_Suitability_Score",
            "EV_Transition_Class"
        ]
    ]
    .sort_values(
        "EV_Transition_Suitability_Score",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

print("\n\nCLASS DISTRIBUTION")
print("=" * 80)

print(
    target_df["EV_Transition_Class"]
    .value_counts()
)

print("\n\nTARGET STATISTICS")
print("=" * 80)

print(
    target_df["EV_Transition_Suitability_Score"]
    .describe()
)

NEW TARGET CREATED
 Depot ID     Depot Name  Economic_Benefit_Score  Operational_Utilization_Score  Route_Feasibility_Score  EV_Transition_Suitability_Score EV_Transition_Class
KSRTC-055           PALA                1.000000                       0.755132                     0.75                         0.851540        EV Preferred
KSRTC-008  CHANGANASSERY                0.996903                       0.755681                     0.75                         0.850465        EV Preferred
KSRTC-037       KOTTAYAM                0.995980                       0.747867                     0.75                         0.847752        EV Preferred
KSRTC-086        VAIKKOM                0.993800                       0.747427                     0.75                         0.846748        EV Preferred
KSRTC-015  EERATTUPETTAH                0.993804                       0.747160                     0.75                         0.846670        EV Preferred
KSRTC-068      PONKUNNAM         

In [6]:
# ============================================================
# CELL 6 — VALIDATE EV TRANSITION TARGET
# ============================================================

# ------------------------------------------------------------
# 1. CLASS DISTRIBUTION
# ------------------------------------------------------------

print("EV TRANSITION CLASS DISTRIBUTION")
print("=" * 70)

class_counts = (
    target_df["EV_Transition_Class"]
    .value_counts()
)

class_percent = (
    target_df["EV_Transition_Class"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percent
})

print(class_summary)


# ------------------------------------------------------------
# 2. SCORE DISTRIBUTION
# ------------------------------------------------------------

print("\n\nSCORE DISTRIBUTION")
print("=" * 70)

print(
    target_df["EV_Transition_Suitability_Score"]
    .describe()
)


# ------------------------------------------------------------
# 3. TOP 15 DEPOTS
# ------------------------------------------------------------

print("\n\nTOP 15 EV TRANSITION CANDIDATES")
print("=" * 70)

top15 = (
    target_df[
        [
            "Depot ID",
            "Depot Name",
            "Economic_Benefit_Score",
            "Operational_Utilization_Score",
            "Route_Feasibility_Score",
            "EV_Transition_Suitability_Score",
            "EV_Transition_Class"
        ]
    ]
    .sort_values(
        "EV_Transition_Suitability_Score",
        ascending=False
    )
    .head(15)
)

print(top15.to_string(index=False))


# ------------------------------------------------------------
# 4. BOTTOM 15 DEPOTS
# ------------------------------------------------------------

print("\n\nBOTTOM 15 EV TRANSITION CANDIDATES")
print("=" * 70)

bottom15 = (
    target_df[
        [
            "Depot ID",
            "Depot Name",
            "Economic_Benefit_Score",
            "Operational_Utilization_Score",
            "Route_Feasibility_Score",
            "EV_Transition_Suitability_Score",
            "EV_Transition_Class"
        ]
    ]
    .sort_values(
        "EV_Transition_Suitability_Score",
        ascending=True
    )
    .head(15)
)

print(bottom15.to_string(index=False))


# ------------------------------------------------------------
# 5. TERRAIN VS FINAL DECISION
# ------------------------------------------------------------

print("\n\nTERRAIN CLASS VS EV TRANSITION DECISION")
print("=" * 70)

terrain_decision = pd.crosstab(
    target_df["Terrain_Class"],
    target_df["EV_Transition_Class"]
)

print(terrain_decision)


# ------------------------------------------------------------
# 6. COMPONENT CORRELATIONS
# ------------------------------------------------------------

print("\n\nCORRELATION BETWEEN TARGET COMPONENTS")
print("=" * 70)

component_cols = [
    "Economic_Benefit_Score",
    "Operational_Utilization_Score",
    "Route_Feasibility_Score",
    "EV_Transition_Suitability_Score"
]

print(
    target_df[component_cols]
    .corr()
    .round(3)
)


# ------------------------------------------------------------
# 7. CHECK WHICH COMPONENT DOMINATES
# ------------------------------------------------------------

print("\n\nAVERAGE COMPONENT SCORES")
print("=" * 70)

print(
    target_df[
        [
            "Economic_Benefit_Score",
            "Operational_Utilization_Score",
            "Route_Feasibility_Score"
        ]
    ]
    .mean()
    .round(3)
)

EV TRANSITION CLASS DISTRIBUTION
                     Count  Percentage
EV_Transition_Class                   
EV Conditional          47       51.09
Diesel Preferred        31       33.70
EV Preferred            14       15.22


SCORE DISTRIBUTION
count    92.000000
mean      0.565985
std       0.149329
min       0.386693
25%       0.401351
50%       0.561176
75%       0.651477
max       0.851540
Name: EV_Transition_Suitability_Score, dtype: float64


TOP 15 EV TRANSITION CANDIDATES
 Depot ID     Depot Name  Economic_Benefit_Score  Operational_Utilization_Score  Route_Feasibility_Score  EV_Transition_Suitability_Score EV_Transition_Class
KSRTC-055           PALA                1.000000                       0.755132                     0.75                         0.851540        EV Preferred
KSRTC-008  CHANGANASSERY                0.996903                       0.755681                     0.75                         0.850465        EV Preferred
KSRTC-037       KOTTAYAM             

In [7]:
# ============================================================
# CELL 7 — VALIDATE TERRAIN AND EV TRANSITION DECISIONS
# ============================================================

print("TERRAIN CLASS VS EV TRANSITION DECISION")
print("=" * 80)

terrain_table = pd.crosstab(
    target_df["Terrain_Class"],
    target_df["EV_Transition_Class"],
    margins=True
)

print(terrain_table)


print("\n\nAVERAGE SUITABILITY BY TERRAIN")
print("=" * 80)

terrain_summary = (
    target_df
    .groupby("Terrain_Class")
    .agg(
        Depots=("Depot ID", "count"),
        Avg_Score=("EV_Transition_Suitability_Score", "mean"),
        Min_Score=("EV_Transition_Suitability_Score", "min"),
        Max_Score=("EV_Transition_Suitability_Score", "max")
    )
    .sort_values("Avg_Score", ascending=False)
)

print(terrain_summary)


print("\n\nDECISION THRESHOLD CHECK")
print("=" * 80)

print("EV Preferred   : Score >= 0.70")
print("EV Conditional : 0.45 <= Score < 0.70")
print("Diesel Preferred: Score < 0.45")


print("\n\nFINAL DEPOT RANKING — TOP 20")
print("=" * 80)

ranking = (
    target_df[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Economic_Benefit_Score",
            "Operational_Utilization_Score",
            "Route_Feasibility_Score",
            "EV_Transition_Suitability_Score",
            "EV_Transition_Class"
        ]
    ]
    .sort_values(
        "EV_Transition_Suitability_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranking["Current_Rank"] = ranking.index + 1

print(
    ranking.head(20).to_string(index=False)
)

TERRAIN CLASS VS EV TRANSITION DECISION
EV_Transition_Class  Diesel Preferred  EV Conditional  EV Preferred  All
Terrain_Class                                                           
Flat                                0               7             0    7
Flat/Rolling                       21              32             4   57
Hilly                               3               0             0    3
Rolling                             0               8            10   18
Steep                               7               0             0    7
All                                31              47            14   92


AVERAGE SUITABILITY BY TERRAIN
               Depots  Avg_Score  Min_Score  Max_Score
Terrain_Class                                         
Rolling            18   0.728459   0.619673   0.851540
Flat                7   0.696521   0.694981   0.697494
Flat/Rolling       57   0.527526   0.386693   0.826973
Hilly               3   0.404755   0.403334   0.405590
Steep        

In [8]:
# ============================================================
# CELL 8 — CREATE MONTHLY EV TRANSITION TARGET
# ============================================================

monthly_df = df.copy()

# Convert Month to datetime
monthly_df["Month"] = pd.to_datetime(monthly_df["Month"])

# ------------------------------------------------------------
# OPERATIONAL FEATURES
# ------------------------------------------------------------

monthly_df["KM_per_Bus"] = (
    monthly_df["Effective KM"] /
    monthly_df["Buses Allocated"]
)

monthly_df["Passengers_per_Bus"] = (
    monthly_df["Passengers"] /
    monthly_df["Buses Allocated"]
)

# ------------------------------------------------------------
# ECONOMIC OPPORTUNITY
# ------------------------------------------------------------

monthly_df["OPEX_Saving_per_Bus"] = (
    monthly_df["Potential EV OPEX Saving (INR)"] /
    monthly_df["Buses Allocated"]
)

# ------------------------------------------------------------
# MERGE TERRAIN
# ------------------------------------------------------------

monthly_df = monthly_df.merge(
    route_df[
        [
            "Depot_ID",
            "Terrain_Class",
            "Terrain_Score"
        ]
    ],
    left_on="Depot ID",
    right_on="Depot_ID",
    how="left"
)

monthly_df.drop(columns=["Depot_ID"], inplace=True)

# ------------------------------------------------------------
# CHECK MERGE
# ------------------------------------------------------------

print("MONTHLY DATA SHAPE:", monthly_df.shape)

print(
    "MISSING TERRAIN:",
    monthly_df["Terrain_Score"].isna().sum()
)

# ------------------------------------------------------------
# TRAIN / TEST PERIOD
# ------------------------------------------------------------

train_mask = monthly_df["Year"] <= 2025
test_mask = monthly_df["Year"] == 2026

# ------------------------------------------------------------
# NORMALIZATION
# IMPORTANT:
# Fit normalization ONLY on 2021–2025 to avoid test leakage.
# ------------------------------------------------------------

def minmax_from_train(series, train_series):
    train_min = train_series.min()
    train_max = train_series.max()

    if train_max == train_min:
        return pd.Series(0.5, index=series.index)

    return (
        (series - train_min) /
        (train_max - train_min)
    ).clip(0, 1)


# Economic score
monthly_df["Economic_Benefit_Score"] = minmax_from_train(
    monthly_df["OPEX_Saving_per_Bus"],
    monthly_df.loc[train_mask, "OPEX_Saving_per_Bus"]
)

# Operational score
km_score = minmax_from_train(
    monthly_df["KM_per_Bus"],
    monthly_df.loc[train_mask, "KM_per_Bus"]
)

passenger_score = minmax_from_train(
    monthly_df["Passengers_per_Bus"],
    monthly_df.loc[train_mask, "Passengers_per_Bus"]
)

monthly_df["Operational_Utilization_Score"] = (
    0.60 * km_score +
    0.40 * passenger_score
)

# Route feasibility comes directly from ROUTE.csv
monthly_df["Route_Feasibility_Score"] = (
    monthly_df["Terrain_Score"]
)

# ------------------------------------------------------------
# FINAL MONTHLY TARGET
# ------------------------------------------------------------

monthly_df["EV_Transition_Suitability_Score"] = (
    0.40 * monthly_df["Economic_Benefit_Score"] +
    0.30 * monthly_df["Operational_Utilization_Score"] +
    0.30 * monthly_df["Route_Feasibility_Score"]
)

monthly_df["EV_Transition_Suitability_Score"] = (
    monthly_df["EV_Transition_Suitability_Score"]
    .clip(0, 1)
)

# ------------------------------------------------------------
# TARGET CLASS
# ------------------------------------------------------------

def classify_transition(score):
    if score >= 0.70:
        return "EV Preferred"
    elif score >= 0.45:
        return "EV Conditional"
    else:
        return "Diesel Preferred"

monthly_df["EV_Transition_Class"] = (
    monthly_df["EV_Transition_Suitability_Score"]
    .apply(classify_transition)
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\nMONTHLY TARGET CREATED")
print("=" * 80)

print(
    monthly_df[
        [
            "Depot ID",
            "Depot Name",
            "Month",
            "Economic_Benefit_Score",
            "Operational_Utilization_Score",
            "Route_Feasibility_Score",
            "EV_Transition_Suitability_Score",
            "EV_Transition_Class"
        ]
    ].head(10).to_string(index=False)
)

print("\n\nTARGET DISTRIBUTION")
print("=" * 80)

print(
    monthly_df["EV_Transition_Class"]
    .value_counts()
)

print("\n\nTARGET SCORE SUMMARY")
print("=" * 80)

print(
    monthly_df["EV_Transition_Suitability_Score"]
    .describe()
)

MONTHLY DATA SHAPE: (5520, 25)
MISSING TERRAIN: 0

MONTHLY TARGET CREATED
 Depot ID Depot Name      Month  Economic_Benefit_Score  Operational_Utilization_Score  Route_Feasibility_Score  EV_Transition_Suitability_Score EV_Transition_Class
KSRTC-001      ADOOR 2021-04-01                0.045461                       0.086283                     0.75                         0.269069    Diesel Preferred
KSRTC-001      ADOOR 2021-05-01                0.047209                       0.092695                     0.75                         0.271692    Diesel Preferred
KSRTC-001      ADOOR 2021-06-01                0.061590                       0.107635                     0.75                         0.281927    Diesel Preferred
KSRTC-001      ADOOR 2021-07-01                0.078879                       0.128400                     0.75                         0.295072    Diesel Preferred
KSRTC-001      ADOOR 2021-08-01                0.030398                       0.076442               

In [10]:
# ============================================================
# CELL 9 — CREATE LAGGED ML DATASET
# Predict NEXT month's EV Transition Suitability
# from PREVIOUS month's depot conditions
# ============================================================

ml_df = monthly_df.copy()

# ------------------------------------------------------------
# 1. CREATE TIME FEATURES
# ------------------------------------------------------------

ml_df["Month_Number"] = ml_df["Month"].dt.month

# Make sure records are in chronological order for each depot
ml_df = (
    ml_df
    .sort_values(["Depot ID", "Month"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. CREATE PREVIOUS-MONTH FEATURES
# ------------------------------------------------------------

lag_features = [
    "Buses Allocated",
    "Schedules Allocated",
    "Effective KM",
    "Passengers",
    "KM_per_Bus",
    "Passengers_per_Bus",
    "OPEX_Saving_per_Bus"
]

for col in lag_features:
    ml_df[f"{col}_Lag1"] = (
        ml_df.groupby("Depot ID")[col].shift(1)
    )

# ------------------------------------------------------------
# 3. REMOVE FIRST MONTH OF EACH DEPOT
# ------------------------------------------------------------

ml_df = ml_df.dropna(
    subset=[f"{col}_Lag1" for col in lag_features]
).copy()

# ------------------------------------------------------------
# 4. DEFINE ML FEATURES
# ------------------------------------------------------------

feature_cols = [
    "Buses Allocated_Lag1",
    "Schedules Allocated_Lag1",
    "Effective KM_Lag1",
    "Passengers_Lag1",
    "KM_per_Bus_Lag1",
    "Passengers_per_Bus_Lag1",
    "OPEX_Saving_per_Bus_Lag1",
    "Terrain_Score",
    "Month_Number"
]

target_col = "EV_Transition_Suitability_Score"

X = ml_df[feature_cols]
y = ml_df[target_col]

# ------------------------------------------------------------
# 5. TIME-BASED TRAIN / TEST SPLIT
# ------------------------------------------------------------

train_mask = ml_df["Year"] <= 2025
test_mask = ml_df["Year"] == 2026

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

# ------------------------------------------------------------
# 6. DISPLAY INFORMATION
# ------------------------------------------------------------

print("ML DATASET SHAPE:", ml_df.shape)

print("\nTRAINING DATA")
print("=" * 70)

print("Rows:", len(X_train))
print(
    "Period:",
    ml_df.loc[train_mask, "Month"].min(),
    "to",
    ml_df.loc[train_mask, "Month"].max()
)

print("\nTEST DATA")
print("=" * 70)

print("Rows:", len(X_test))
print(
    "Period:",
    ml_df.loc[test_mask, "Month"].min(),
    "to",
    ml_df.loc[test_mask, "Month"].max()
)

print("\nFEATURES")
print("=" * 70)

for i, feature in enumerate(feature_cols, 1):
    print(f"{i}. {feature}")

print("\nTARGET")
print("=" * 70)
print(target_col)

print("\nTARGET MEAN")
print("=" * 70)

print("Train:", round(y_train.mean(), 4))
print("Test :", round(y_test.mean(), 4))

print("\nTARGET CLASS DISTRIBUTION")
print("=" * 70)

print(
    ml_df.loc[train_mask, "EV_Transition_Class"]
    .value_counts()
)

print("\nTEST CLASS DISTRIBUTION")
print("=" * 70)

print(
    ml_df.loc[test_mask, "EV_Transition_Class"]
    .value_counts()
)

ML DATASET SHAPE: (5428, 38)

TRAINING DATA
Rows: 5152
Period: 2021-05-01 00:00:00 to 2025-12-01 00:00:00

TEST DATA
Rows: 276
Period: 2026-01-01 00:00:00 to 2026-03-01 00:00:00

FEATURES
1. Buses Allocated_Lag1
2. Schedules Allocated_Lag1
3. Effective KM_Lag1
4. Passengers_Lag1
5. KM_per_Bus_Lag1
6. Passengers_per_Bus_Lag1
7. OPEX_Saving_per_Bus_Lag1
8. Terrain_Score
9. Month_Number

TARGET
EV_Transition_Suitability_Score

TARGET MEAN
Train: 0.5486
Test : 0.5587

TARGET CLASS DISTRIBUTION
EV_Transition_Class
EV Conditional      3568
Diesel Preferred    1192
EV Preferred         392
Name: count, dtype: int64

TEST CLASS DISTRIBUTION
EV_Transition_Class
EV Conditional      246
Diesel Preferred     30
Name: count, dtype: int64


In [11]:
# ============================================================
# CELL 10 — RANDOM FOREST MODEL
# Predict next-month EV Transition Suitability Score
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ------------------------------------------------------------
# 1. CREATE MODEL
# ------------------------------------------------------------

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# 2. TRAIN
# ------------------------------------------------------------

rf_model.fit(X_train, y_train)

# ------------------------------------------------------------
# 3. PREDICT 2026
# ------------------------------------------------------------

y_pred = rf_model.predict(X_test)

# Keep predictions within valid score range
y_pred = np.clip(y_pred, 0, 1)

# ------------------------------------------------------------
# 4. EVALUATION
# ------------------------------------------------------------

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("RANDOM FOREST RESULTS")
print("=" * 70)

print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")


# ------------------------------------------------------------
# 5. NAIVE PREVIOUS-MONTH BASELINE
# ------------------------------------------------------------

baseline_pred = (
    ml_df.loc[test_mask, target_col]
    .shift(0)
)

# The actual previous-month target is already available
# through the lagged monthly records.
#
# Build it directly from the original monthly target.

baseline_source = (
    monthly_df[
        ["Depot ID", "Month", target_col]
    ]
    .sort_values(["Depot ID", "Month"])
    .copy()
)

baseline_source["Previous_Month_Score"] = (
    baseline_source
    .groupby("Depot ID")[target_col]
    .shift(1)
)

baseline = baseline_source[
    baseline_source["Month"].dt.year == 2026
]["Previous_Month_Score"].values

actual = y_test.values

valid_baseline = ~np.isnan(baseline)

baseline_mae = mean_absolute_error(
    actual[valid_baseline],
    baseline[valid_baseline]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        actual[valid_baseline],
        baseline[valid_baseline]
    )
)

baseline_r2 = r2_score(
    actual[valid_baseline],
    baseline[valid_baseline]
)

print("\nNAIVE PREVIOUS-MONTH BASELINE")
print("=" * 70)

print(f"MAE  : {baseline_mae:.6f}")
print(f"RMSE : {baseline_rmse:.6f}")
print(f"R²   : {baseline_r2:.6f}")


# ------------------------------------------------------------
# 6. MODEL IMPROVEMENT
# ------------------------------------------------------------

mae_improvement = (
    (baseline_mae - mae) /
    baseline_mae
) * 100

rmse_improvement = (
    (baseline_rmse - rmse) /
    baseline_rmse
) * 100

print("\nMODEL IMPROVEMENT OVER NAIVE BASELINE")
print("=" * 70)

print(f"MAE improvement  : {mae_improvement:.2f}%")
print(f"RMSE improvement : {rmse_improvement:.2f}%")


# ------------------------------------------------------------
# 7. FEATURE IMPORTANCE
# ------------------------------------------------------------

importance_df = (
    pd.DataFrame({
        "Feature": feature_cols,
        "Importance": rf_model.feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print("\nFEATURE IMPORTANCE")
print("=" * 70)

print(
    importance_df.to_string(index=False)
)

RANDOM FOREST RESULTS
MAE  : 0.004025
RMSE : 0.004970
R²   : 0.993582

NAIVE PREVIOUS-MONTH BASELINE
MAE  : 0.004782
RMSE : 0.006145
R²   : 0.990185

MODEL IMPROVEMENT OVER NAIVE BASELINE
MAE improvement  : 15.82%
RMSE improvement : 19.13%

FEATURE IMPORTANCE
                 Feature  Importance
         KM_per_Bus_Lag1    0.313776
OPEX_Saving_per_Bus_Lag1    0.306686
           Terrain_Score    0.242300
            Month_Number    0.070676
 Passengers_per_Bus_Lag1    0.042509
       Effective KM_Lag1    0.013635
         Passengers_Lag1    0.006214
Schedules Allocated_Lag1    0.002311
    Buses Allocated_Lag1    0.001894


In [12]:
# ============================================================
# CELL 11 — 2026 DEPOT EV TRANSITION PREDICTION & RANKING
# ============================================================

# Create prediction dataframe
prediction_df = ml_df.loc[test_mask, [
    "Depot ID",
    "Depot Name",
    "District",
    "Month",
    "Terrain_Class",
    "Terrain_Score"
]].copy()

prediction_df["Actual_Score"] = y_test.values
prediction_df["Predicted_Score"] = y_pred

# ------------------------------------------------------------
# CLASSIFY PREDICTED SCORE
# ------------------------------------------------------------

def predicted_class(score):
    if score >= 0.70:
        return "EV Preferred"
    elif score >= 0.45:
        return "EV Conditional"
    else:
        return "Diesel Preferred"

prediction_df["Predicted_Class"] = (
    prediction_df["Predicted_Score"]
    .apply(predicted_class)
)

# ------------------------------------------------------------
# DEPOT-LEVEL AGGREGATION
# ------------------------------------------------------------

depot_predictions = (
    prediction_df
    .groupby(
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Terrain_Score"
        ]
    )
    .agg(
        Avg_Predicted_Score=("Predicted_Score", "mean"),
        Avg_Actual_Score=("Actual_Score", "mean"),
        Min_Predicted_Score=("Predicted_Score", "min"),
        Max_Predicted_Score=("Predicted_Score", "max")
    )
    .reset_index()
)

# ------------------------------------------------------------
# FINAL DECISION
# ------------------------------------------------------------

depot_predictions["EV_Transition_Decision"] = (
    depot_predictions["Avg_Predicted_Score"]
    .apply(predicted_class)
)

# ------------------------------------------------------------
# RANK DEPOTS
# ------------------------------------------------------------

depot_predictions = (
    depot_predictions
    .sort_values(
        "Avg_Predicted_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

depot_predictions["EV_Transition_Rank"] = (
    depot_predictions.index + 1
)

# ------------------------------------------------------------
# DISPLAY TOP 20
# ------------------------------------------------------------

print("TOP 20 DEPOTS — PREDICTED EV TRANSITION SUITABILITY")
print("=" * 100)

print(
    depot_predictions[
        [
            "EV_Transition_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Avg_Predicted_Score",
            "Avg_Actual_Score",
            "EV_Transition_Decision"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# DISPLAY BOTTOM 15
# ------------------------------------------------------------

print("\n\nBOTTOM 15 DEPOTS")
print("=" * 100)

print(
    depot_predictions[
        [
            "EV_Transition_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Avg_Predicted_Score",
            "Avg_Actual_Score",
            "EV_Transition_Decision"
        ]
    ]
    .tail(15)
    .to_string(index=False)
)

# ------------------------------------------------------------
# DECISION DISTRIBUTION
# ------------------------------------------------------------

print("\n\nFINAL PREDICTED DECISION DISTRIBUTION")
print("=" * 100)

print(
    depot_predictions["EV_Transition_Decision"]
    .value_counts()
)

# ------------------------------------------------------------
# TERRAIN VS PREDICTED DECISION
# ------------------------------------------------------------

print("\n\nTERRAIN VS PREDICTED DECISION")
print("=" * 100)

print(
    pd.crosstab(
        depot_predictions["Terrain_Class"],
        depot_predictions["EV_Transition_Decision"]
    )
)

TOP 20 DEPOTS — PREDICTED EV TRANSITION SUITABILITY
 EV_Transition_Rank  Depot ID      Depot Name           District Terrain_Class  Avg_Predicted_Score  Avg_Actual_Score EV_Transition_Decision
                  1 KSRTC-046     MAVELIKKARA          Alappuzha          Flat             0.615876          0.619197         EV Conditional
                  2 KSRTC-010       CHENGANUR          Alappuzha          Flat             0.615544          0.619446         EV Conditional
                  3 KSRTC-002       ALAPPUZHA          Alappuzha          Flat             0.615387          0.615373         EV Conditional
                  4 KSRTC-019        HARIPPAD          Alappuzha          Flat             0.615208          0.612773         EV Conditional
                  5 KSRTC-029      KAYAMKULAM          Alappuzha          Flat             0.615119          0.613498         EV Conditional
                  6 KSRTC-011       CHERTHALA          Alappuzha          Flat             0.614818   

In [13]:
# ============================================================
# CELL 12 — REMOVE REDUNDANT FEATURES & RETRAIN CLEAN MODEL
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ------------------------------------------------------------
# IMPORTANT:
# OPEX_Saving_per_Bus = 24 × KM_per_Bus in our dataset.
# Therefore, they are mathematically redundant.
#
# We keep KM_per_Bus_Lag1 and remove
# OPEX_Saving_per_Bus_Lag1.
# ------------------------------------------------------------

clean_features = [
    "Buses Allocated_Lag1",
    "Schedules Allocated_Lag1",
    "Effective KM_Lag1",
    "Passengers_Lag1",
    "KM_per_Bus_Lag1",
    "Passengers_per_Bus_Lag1",
    "Terrain_Score",
    "Month_Number"
]

X_clean = ml_df[clean_features]
y_clean = ml_df[target_col]

X_clean_train = X_clean.loc[train_mask]
y_clean_train = y_clean.loc[train_mask]

X_clean_test = X_clean.loc[test_mask]
y_clean_test = y_clean.loc[test_mask]

# ------------------------------------------------------------
# TRAIN CLEAN RANDOM FOREST
# ------------------------------------------------------------

clean_rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

clean_rf_model.fit(
    X_clean_train,
    y_clean_train
)

# ------------------------------------------------------------
# PREDICTIONS
# ------------------------------------------------------------

clean_pred = clean_rf_model.predict(X_clean_test)

clean_pred = np.clip(clean_pred, 0, 1)

# ------------------------------------------------------------
# EVALUATION
# ------------------------------------------------------------

clean_mae = mean_absolute_error(
    y_clean_test,
    clean_pred
)

clean_rmse = np.sqrt(
    mean_squared_error(
        y_clean_test,
        clean_pred
    )
)

clean_r2 = r2_score(
    y_clean_test,
    clean_pred
)

print("CLEAN RANDOM FOREST RESULTS")
print("=" * 70)

print(f"MAE  : {clean_mae:.6f}")
print(f"RMSE : {clean_rmse:.6f}")
print(f"R²   : {clean_r2:.6f}")


# ------------------------------------------------------------
# COMPARE WITH PREVIOUS MODEL
# ------------------------------------------------------------

print("\n\nPREVIOUS MODEL VS CLEAN MODEL")
print("=" * 70)

print(
    f"Previous MAE  : {mae:.6f}"
)

print(
    f"Clean MAE     : {clean_mae:.6f}"
)

print(
    f"Previous RMSE : {rmse:.6f}"
)

print(
    f"Clean RMSE    : {clean_rmse:.6f}"
)

print(
    f"Previous R²   : {r2:.6f}"
)

print(
    f"Clean R²      : {clean_r2:.6f}"
)


# ------------------------------------------------------------
# FEATURE IMPORTANCE
# ------------------------------------------------------------

clean_importance = (
    pd.DataFrame({
        "Feature": clean_features,
        "Importance": clean_rf_model.feature_importances_
    })
    .sort_values(
        "Importance",
        ascending=False
    )
)

print("\n\nCLEAN MODEL FEATURE IMPORTANCE")
print("=" * 70)

print(
    clean_importance.to_string(index=False)
)


# ------------------------------------------------------------
# CHECK REDUNDANCY
# ------------------------------------------------------------

print("\n\nCORRELATION CHECK")
print("=" * 70)

print(
    X_clean.corr()
    .round(3)
)

CLEAN RANDOM FOREST RESULTS
MAE  : 0.004021
RMSE : 0.004966
R²   : 0.993591


PREVIOUS MODEL VS CLEAN MODEL
Previous MAE  : 0.004025
Clean MAE     : 0.004021
Previous RMSE : 0.004970
Clean RMSE    : 0.004966
Previous R²   : 0.993582
Clean R²      : 0.993591


CLEAN MODEL FEATURE IMPORTANCE
                 Feature  Importance
         KM_per_Bus_Lag1    0.620129
           Terrain_Score    0.242304
            Month_Number    0.070684
 Passengers_per_Bus_Lag1    0.042541
       Effective KM_Lag1    0.013784
         Passengers_Lag1    0.006225
Schedules Allocated_Lag1    0.002378
    Buses Allocated_Lag1    0.001955


CORRELATION CHECK
                          Buses Allocated_Lag1  Schedules Allocated_Lag1  \
Buses Allocated_Lag1                     1.000                     0.835   
Schedules Allocated_Lag1                 0.835                     1.000   
Effective KM_Lag1                        0.861                     0.747   
Passengers_Lag1                          0.837      

In [14]:
# ============================================================
# CELL 13 — VALIDATE 2026 PREDICTIONS & THRESHOLDS
# ============================================================

# ------------------------------------------------------------
# 1. CREATE 2026 PREDICTION DATA
# ------------------------------------------------------------

validation_df = ml_df.loc[test_mask, [
    "Depot ID",
    "Depot Name",
    "District",
    "Month",
    "Terrain_Class",
    "Terrain_Score"
]].copy()

validation_df["Actual_Score"] = y_clean_test.values
validation_df["Predicted_Score"] = clean_pred

# Prediction error
validation_df["Absolute_Error"] = (
    validation_df["Actual_Score"] -
    validation_df["Predicted_Score"]
).abs()

# ------------------------------------------------------------
# 2. PREDICTED SCORE DISTRIBUTION
# ------------------------------------------------------------

print("2026 PREDICTED SCORE DISTRIBUTION")
print("=" * 80)

print(
    validation_df["Predicted_Score"].describe()
)

# ------------------------------------------------------------
# 3. ACTUAL VS PREDICTED
# ------------------------------------------------------------

print("\n\nACTUAL VS PREDICTED SCORE")
print("=" * 80)

comparison = (
    validation_df[
        [
            "Actual_Score",
            "Predicted_Score",
            "Absolute_Error"
        ]
    ]
    .describe()
)

print(comparison)

# ------------------------------------------------------------
# 4. CHECK CURRENT THRESHOLDS
# ------------------------------------------------------------

print("\n\nCURRENT DECISION THRESHOLDS")
print("=" * 80)

print("EV Preferred    : >= 0.70")
print("EV Conditional  : 0.45 – <0.70")
print("Diesel Preferred: < 0.45")

print("\n2026 PREDICTED DECISION COUNTS")
print("-" * 80)

def classify_score(score):
    if score >= 0.70:
        return "EV Preferred"
    elif score >= 0.45:
        return "EV Conditional"
    else:
        return "Diesel Preferred"

validation_df["Predicted_Class"] = (
    validation_df["Predicted_Score"]
    .apply(classify_score)
)

print(
    validation_df["Predicted_Class"]
    .value_counts()
)

# ------------------------------------------------------------
# 5. ACTUAL TARGET DECISION COUNTS
# ------------------------------------------------------------

validation_df["Actual_Class"] = (
    validation_df["Actual_Score"]
    .apply(classify_score)
)

print("\n2026 ACTUAL TARGET DECISION COUNTS")
print("-" * 80)

print(
    validation_df["Actual_Class"]
    .value_counts()
)

# ------------------------------------------------------------
# 6. TOP 20 PREDICTIONS
# ------------------------------------------------------------

print("\n\nTOP 20 PREDICTED DEPOTS")
print("=" * 100)

top20 = (
    validation_df
    .groupby(
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class"
        ]
    )
    .agg(
        Predicted_Score=("Predicted_Score", "mean"),
        Actual_Score=("Actual_Score", "mean"),
        Error=("Absolute_Error", "mean")
    )
    .sort_values(
        "Predicted_Score",
        ascending=False
    )
    .head(20)
)

print(
    top20.to_string()
)

# ------------------------------------------------------------
# 7. LOWEST 15 PREDICTIONS
# ------------------------------------------------------------

print("\n\nBOTTOM 15 PREDICTED DEPOTS")
print("=" * 100)

bottom15 = (
    validation_df
    .groupby(
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class"
        ]
    )
    .agg(
        Predicted_Score=("Predicted_Score", "mean"),
        Actual_Score=("Actual_Score", "mean"),
        Error=("Absolute_Error", "mean")
    )
    .sort_values(
        "Predicted_Score",
        ascending=True
    )
    .head(15)
)

print(
    bottom15.to_string()
)

# ------------------------------------------------------------
# 8. CHECK WHETHER PREDICTIONS ARE SYSTEMATICALLY BIASED
# ------------------------------------------------------------

print("\n\nPREDICTION BIAS")
print("=" * 80)

bias = (
    validation_df["Predicted_Score"] -
    validation_df["Actual_Score"]
)

print("Mean prediction error:", round(bias.mean(), 6))
print("Median prediction error:", round(bias.median(), 6))

if bias.mean() > 0:
    print("Model tendency: slight OVER-PREDICTION")
elif bias.mean() < 0:
    print("Model tendency: slight UNDER-PREDICTION")
else:
    print("Model tendency: no systematic bias")

# ------------------------------------------------------------
# 9. DEPOT RANKING
# ------------------------------------------------------------

print("\n\nNUMBER OF DEPOTS RANKED:",
      validation_df["Depot ID"].nunique())

2026 PREDICTED SCORE DISTRIBUTION
count    276.000000
mean       0.558487
std        0.061709
min        0.372442
25%        0.541578
50%        0.585690
75%        0.585731
max        0.617117
Name: Predicted_Score, dtype: float64


ACTUAL VS PREDICTED SCORE
       Actual_Score  Predicted_Score  Absolute_Error
count    276.000000       276.000000      276.000000
mean       0.558708         0.558487        0.004021
std        0.062144         0.061709        0.002919
min        0.364602         0.372442        0.000015
25%        0.544926         0.541578        0.001926
50%        0.583555         0.585690        0.003631
75%        0.588932         0.585731        0.005700
max        0.622559         0.617117        0.024682


CURRENT DECISION THRESHOLDS
EV Preferred    : >= 0.70
EV Conditional  : 0.45 – <0.70
Diesel Preferred: < 0.45

2026 PREDICTED DECISION COUNTS
--------------------------------------------------------------------------------
Predicted_Class
EV Conditional      24

In [15]:
# ============================================================
# CELL 14 — FINAL 2026 DEPOT RANKING
# USING CLEAN RANDOM FOREST MODEL
# ============================================================

# ------------------------------------------------------------
# 1. CREATE 2026 PREDICTION DATAFRAME
# ------------------------------------------------------------

final_pred_df = ml_df.loc[
    test_mask,
    [
        "Depot ID",
        "Depot Name",
        "District",
        "Month",
        "Terrain_Class",
        "Terrain_Score"
    ]
].copy()

final_pred_df["Actual_Score"] = y_clean_test.values
final_pred_df["Predicted_Score"] = clean_pred

# ------------------------------------------------------------
# 2. CALCULATE PREDICTION ERROR
# ------------------------------------------------------------

final_pred_df["Absolute_Error"] = (
    final_pred_df["Actual_Score"] -
    final_pred_df["Predicted_Score"]
).abs()

# ------------------------------------------------------------
# 3. CONVERT PREDICTED SCORE TO DECISION
# ------------------------------------------------------------

def final_decision(score):

    if score >= 0.70:
        return "EV Preferred"

    elif score >= 0.45:
        return "EV Conditional"

    else:
        return "Diesel Preferred"


final_pred_df["Predicted_Decision"] = (
    final_pred_df["Predicted_Score"]
    .apply(final_decision)
)

# ------------------------------------------------------------
# 4. AGGREGATE JAN–MAR 2026 BY DEPOT
# ------------------------------------------------------------

final_depot_ranking = (
    final_pred_df
    .groupby(
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Terrain_Score"
        ]
    )
    .agg(
        Predicted_EV_Suitability=(
            "Predicted_Score",
            "mean"
        ),
        Actual_EV_Suitability=(
            "Actual_Score",
            "mean"
        ),
        Prediction_Error=(
            "Absolute_Error",
            "mean"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. FINAL DECISION
# ------------------------------------------------------------

final_depot_ranking["EV_Transition_Decision"] = (
    final_depot_ranking[
        "Predicted_EV_Suitability"
    ].apply(final_decision)
)

# ------------------------------------------------------------
# 6. RANK ALL 92 DEPOTS
# ------------------------------------------------------------

final_depot_ranking = (
    final_depot_ranking
    .sort_values(
        "Predicted_EV_Suitability",
        ascending=False
    )
    .reset_index(drop=True)
)

final_depot_ranking["EV_Transition_Rank"] = (
    final_depot_ranking.index + 1
)

# ------------------------------------------------------------
# 7. DISPLAY TOP 20
# ------------------------------------------------------------

print("FINAL 2026 EV TRANSITION DEPOT RANKING")
print("=" * 110)

print(
    final_depot_ranking[
        [
            "EV_Transition_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Predicted_EV_Suitability",
            "Actual_EV_Suitability",
            "EV_Transition_Decision"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. DISPLAY BOTTOM 15
# ------------------------------------------------------------

print("\n\nBOTTOM 15 DEPOTS")
print("=" * 110)

print(
    final_depot_ranking[
        [
            "EV_Transition_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Predicted_EV_Suitability",
            "Actual_EV_Suitability",
            "EV_Transition_Decision"
        ]
    ]
    .tail(15)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. FINAL DECISION DISTRIBUTION
# ------------------------------------------------------------

print("\n\nFINAL DECISION DISTRIBUTION")
print("=" * 80)

print(
    final_depot_ranking[
        "EV_Transition_Decision"
    ].value_counts()
)

# ------------------------------------------------------------
# 10. VERIFY ALL DEPOTS ARE PRESENT
# ------------------------------------------------------------

print("\n\nDEPOT COVERAGE")
print("=" * 80)

print(
    "Unique depots ranked:",
    final_depot_ranking["Depot ID"].nunique()
)

print(
    "Expected depots: 92"
)

FINAL 2026 EV TRANSITION DEPOT RANKING
 EV_Transition_Rank  Depot ID      Depot Name           District Terrain_Class  Predicted_EV_Suitability  Actual_EV_Suitability EV_Transition_Decision
                  1 KSRTC-046     MAVELIKKARA          Alappuzha          Flat                  0.615878               0.619197         EV Conditional
                  2 KSRTC-010       CHENGANUR          Alappuzha          Flat                  0.615554               0.619446         EV Conditional
                  3 KSRTC-002       ALAPPUZHA          Alappuzha          Flat                  0.615395               0.615373         EV Conditional
                  4 KSRTC-019        HARIPPAD          Alappuzha          Flat                  0.615210               0.612773         EV Conditional
                  5 KSRTC-029      KAYAMKULAM          Alappuzha          Flat                  0.615115               0.613498         EV Conditional
                  6 KSRTC-011       CHERTHALA          

In [16]:
# ============================================================
# CELL 15 — HISTORICAL EV TRANSITION SUITABILITY ANALYSIS
# ============================================================

print("HISTORICAL EV TRANSITION SUITABILITY ANALYSIS")
print("=" * 85)

# ------------------------------------------------------------
# 1. OVERALL SCORE DISTRIBUTION
# ------------------------------------------------------------

print("\nOVERALL SCORE DISTRIBUTION")
print("-" * 85)

print(
    monthly_df["EV_Transition_Suitability_Score"]
    .describe(
        percentiles=[
            0.10,
            0.20,
            0.25,
            0.50,
            0.75,
            0.80,
            0.90,
            0.95
        ]
    )
)


# ------------------------------------------------------------
# 2. YEAR-WISE SCORE DISTRIBUTION
# ------------------------------------------------------------

print("\n\nYEAR-WISE SUITABILITY")
print("-" * 85)

year_summary = (
    monthly_df
    .groupby("Year")[
        "EV_Transition_Suitability_Score"
    ]
    .agg(
        Count="count",
        Mean="mean",
        Minimum="min",
        Median="median",
        Maximum="max"
    )
    .round(4)
)

print(year_summary)


# ------------------------------------------------------------
# 3. YEAR-WISE CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\n\nYEAR-WISE EV TRANSITION CLASS")
print("-" * 85)

year_class = pd.crosstab(
    monthly_df["Year"],
    monthly_df["EV_Transition_Class"]
)

print(year_class)


# ------------------------------------------------------------
# 4. DEPOT-LEVEL HISTORICAL AVERAGE
# ------------------------------------------------------------

print("\n\nDEPOT-LEVEL HISTORICAL AVERAGE")
print("-" * 85)

historical_depot = (
    monthly_df
    .groupby(
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class"
        ]
    )
    .agg(
        Historical_Avg_Score=(
            "EV_Transition_Suitability_Score",
            "mean"
        ),
        Historical_Max_Score=(
            "EV_Transition_Suitability_Score",
            "max"
        ),
        Historical_Min_Score=(
            "EV_Transition_Suitability_Score",
            "min"
        )
    )
    .reset_index()
    .sort_values(
        "Historical_Avg_Score",
        ascending=False
    )
)

print(
    historical_depot.head(20)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. 2026 VS HISTORICAL DEPOT SCORE
# ------------------------------------------------------------

print("\n\n2026 VS HISTORICAL DEPOT SCORE")
print("-" * 85)

historical_depot_compare = historical_depot.merge(
    final_depot_ranking[
        [
            "Depot ID",
            "Predicted_EV_Suitability"
        ]
    ],
    on="Depot ID",
    how="inner"
)

historical_depot_compare["Change_From_Historical"] = (
    historical_depot_compare["Predicted_EV_Suitability"]
    -
    historical_depot_compare["Historical_Avg_Score"]
)

print(
    historical_depot_compare[
        [
            "Depot ID",
            "Depot Name",
            "Terrain_Class",
            "Historical_Avg_Score",
            "Predicted_EV_Suitability",
            "Change_From_Historical"
        ]
    ]
    .sort_values(
        "Predicted_EV_Suitability",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 6. SCORE PERCENTILES
# ------------------------------------------------------------

print("\n\nSUITABILITY PERCENTILES")
print("-" * 85)

percentiles = (
    monthly_df[
        "EV_Transition_Suitability_Score"
    ]
    .quantile(
        [0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

for p, value in percentiles.items():
    print(
        f"{int(p * 100):>3}th percentile : {value:.4f}"
    )

HISTORICAL EV TRANSITION SUITABILITY ANALYSIS

OVERALL SCORE DISTRIBUTION
-------------------------------------------------------------------------------------
count    5520.000000
mean        0.546008
std         0.123366
min         0.147214
10%         0.369112
20%         0.424252
25%         0.460935
50%         0.583380
75%         0.618550
80%         0.631177
90%         0.673691
95%         0.720276
max         0.914777
Name: EV_Transition_Suitability_Score, dtype: float64


YEAR-WISE SUITABILITY
-------------------------------------------------------------------------------------
      Count    Mean  Minimum  Median  Maximum
Year                                         
2021    828  0.3645   0.1552  0.3754   0.4877
2022   1104  0.5520   0.1472  0.6139   0.7772
2023   1104  0.6294   0.3598  0.6345   0.9087
2024   1104  0.5773   0.3642  0.5845   0.9148
2025   1104  0.5583   0.3646  0.5832   0.6230
2026    276  0.5587   0.3646  0.5836   0.6226


YEAR-WISE EV TRANSITION CLASS
---

In [17]:
# ============================================================
# CELL 16 — FINAL DEPOT EV TRANSITION DECISION TABLE
# ============================================================

# ------------------------------------------------------------
# 1. START WITH THE CLEAN MODEL PREDICTIONS
# ------------------------------------------------------------

final_depots = final_depot_ranking.copy()

# ------------------------------------------------------------
# 2. MERGE DEPOT-LEVEL ECONOMIC & OPERATIONAL INFORMATION
# ------------------------------------------------------------

depot_info = target_df[
    [
        "Depot ID",
        "Depot Name",
        "District",
        "Avg_KM_per_Bus",
        "Avg_Passengers_per_Bus",
        "Avg_Daily_KM_per_Bus",
        "Avg_OPEX_Saving",
        "Terrain_Class",
        "Terrain_Score",
        "Economic_Benefit_Score",
        "Operational_Utilization_Score",
        "Route_Feasibility_Score",
        "EV_Transition_Suitability_Score"
    ]
].copy()

# Avoid duplicate columns during merge
depot_info = depot_info.rename(
    columns={
        "EV_Transition_Suitability_Score":
        "Historical_Transition_Score"
    }
)

final_depots = final_depots.merge(
    depot_info,
    on=[
        "Depot ID",
        "Depot Name",
        "District",
        "Terrain_Class",
        "Terrain_Score"
    ],
    how="left"
)

# ------------------------------------------------------------
# 3. FINAL DECISION
# ------------------------------------------------------------

def final_ev_decision(score):

    if score >= 0.70:
        return "EV Preferred"

    elif score >= 0.45:
        return "EV Conditional"

    else:
        return "Diesel Preferred"


final_depots["Final_EV_Decision"] = (
    final_depots["Predicted_EV_Suitability"]
    .apply(final_ev_decision)
)

# ------------------------------------------------------------
# 4. FINAL RANK
# ------------------------------------------------------------

final_depots = (
    final_depots
    .sort_values(
        "Predicted_EV_Suitability",
        ascending=False
    )
    .reset_index(drop=True)
)

final_depots["Final_EV_Rank"] = (
    final_depots.index + 1
)

# ------------------------------------------------------------
# 5. SELECT FINAL COLUMNS
# ------------------------------------------------------------

final_depots = final_depots[
    [
        "Final_EV_Rank",
        "Depot ID",
        "Depot Name",
        "District",
        "Terrain_Class",
        "Terrain_Score",

        "Predicted_EV_Suitability",
        "Historical_Transition_Score",

        "Avg_KM_per_Bus",
        "Avg_Passengers_per_Bus",
        "Avg_Daily_KM_per_Bus",
        "Avg_OPEX_Saving",

        "Economic_Benefit_Score",
        "Operational_Utilization_Score",
        "Route_Feasibility_Score",

        "Final_EV_Decision"
    ]
]

# ------------------------------------------------------------
# 6. DISPLAY FINAL TOP 20
# ------------------------------------------------------------

print("FINAL EV TRANSITION DEPOT RANKING")
print("=" * 110)

print(
    final_depots.head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. DECISION DISTRIBUTION
# ------------------------------------------------------------

print("\n\nFINAL EV TRANSITION DECISION DISTRIBUTION")
print("=" * 80)

print(
    final_depots["Final_EV_Decision"]
    .value_counts()
)

# ------------------------------------------------------------
# 8. DEPOT COVERAGE
# ------------------------------------------------------------

print("\n\nDEPOT COVERAGE")
print("=" * 80)

print(
    "Depots ranked:",
    final_depots["Depot ID"].nunique()
)

print(
    "Expected:",
    92
)

FINAL EV TRANSITION DEPOT RANKING
 Final_EV_Rank  Depot ID      Depot Name           District Terrain_Class  Terrain_Score  Predicted_EV_Suitability  Historical_Transition_Score  Avg_KM_per_Bus  Avg_Passengers_per_Bus  Avg_Daily_KM_per_Bus  Avg_OPEX_Saving  Economic_Benefit_Score  Operational_Utilization_Score  Route_Feasibility_Score Final_EV_Decision
             1 KSRTC-046     MAVELIKKARA          Alappuzha          Flat            1.0                  0.615878                     0.697494     6880.462290             9984.126747            226.228964        7814627.2                0.531246                       0.616651                      1.0    EV Conditional
             2 KSRTC-010       CHENGANUR          Alappuzha          Flat            1.0                  0.615554                     0.694981     6876.537251             9952.584268            226.115989        7814631.2                0.529468                       0.610646                      1.0    EV Conditional
   

In [18]:
# ============================================================
# CELL 17 — INTEGRATE EV IMPACT ANALYSIS
# ============================================================

# ------------------------------------------------------------
# 1. COPY FINAL DEPOT RANKING
# ------------------------------------------------------------

final_output = final_depots.copy()

# ------------------------------------------------------------
# 2. GET DEPOT-LEVEL HISTORICAL IMPACT
# ------------------------------------------------------------

impact_depot = (
    monthly_df
    .groupby(
        [
            "Depot ID",
            "Depot Name",
            "District"
        ]
    )
    .agg(
        Total_KM=("Effective KM", "sum"),
        Total_Diesel_Litres=(
            "Estimated Diesel Litres",
            "sum"
        ),
        Total_CO2_Tonnes=(
            "Estimated CO2 (Tonnes)",
            "sum"
        ),
        Total_OPEX_Saving_INR=(
            "Potential EV OPEX Saving (INR)",
            "sum"
        ),
        Total_EV_Energy_MWh=(
            "Estimated EV Energy (MWh)",
            "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. MERGE IMPACT DATA
# ------------------------------------------------------------

final_output = final_output.merge(
    impact_depot,
    on=[
        "Depot ID",
        "Depot Name",
        "District"
    ],
    how="left"
)

# ------------------------------------------------------------
# 4. CONVERT ENERGY TO GWh
# ------------------------------------------------------------

final_output["Total_EV_Energy_GWh"] = (
    final_output["Total_EV_Energy_MWh"] / 1000
)

# ------------------------------------------------------------
# 5. OPEX SAVING IN CRORE
# ------------------------------------------------------------

final_output["Total_OPEX_Saving_Crore"] = (
    final_output["Total_OPEX_Saving_INR"] / 1e7
)

# ------------------------------------------------------------
# 6. CO2 REDUCTION
# ------------------------------------------------------------
# Under the project's full diesel-to-EV transition scenario,
# avoided diesel tailpipe CO2 is treated as potential reduction.

final_output["Potential_CO2_Reduction_Tonnes"] = (
    final_output["Total_CO2_Tonnes"]
)

# ------------------------------------------------------------
# 7. PRIORITY CATEGORY
# ------------------------------------------------------------
# This is separate from the engineering decision.
# It tells us where the depot ranks among all 92 depots.

def priority_category(rank):

    if rank <= 10:
        return "High Priority"

    elif rank <= 30:
        return "Medium Priority"

    else:
        return "Lower Priority"


final_output["Transition_Priority"] = (
    final_output["Final_EV_Rank"]
    .apply(priority_category)
)

# ------------------------------------------------------------
# 8. FINAL COLUMN ORDER
# ------------------------------------------------------------

final_output = final_output[
    [
        "Final_EV_Rank",
        "Depot ID",
        "Depot Name",
        "District",
        "Terrain_Class",
        "Terrain_Score",

        "Predicted_EV_Suitability",
        "Historical_Transition_Score",

        "Final_EV_Decision",
        "Transition_Priority",

        "Avg_KM_per_Bus",
        "Avg_Passengers_per_Bus",
        "Avg_Daily_KM_per_Bus",

        "Economic_Benefit_Score",
        "Operational_Utilization_Score",
        "Route_Feasibility_Score",

        "Total_KM",
        "Total_Diesel_Litres",
        "Total_CO2_Tonnes",
        "Potential_CO2_Reduction_Tonnes",

        "Total_EV_Energy_GWh",
        "Total_OPEX_Saving_INR",
        "Total_OPEX_Saving_Crore"
    ]
]

# ------------------------------------------------------------
# 9. DISPLAY TOP 20
# ------------------------------------------------------------

print("FINAL INTEGRATED EV TRANSITION ANALYSIS")
print("=" * 120)

print(
    final_output.head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("\n\nFINAL SUMMARY")
print("=" * 80)

print(
    "Total depots:",
    len(final_output)
)

print(
    "EV Preferred:",
    (final_output["Final_EV_Decision"] == "EV Preferred").sum()
)

print(
    "EV Conditional:",
    (final_output["Final_EV_Decision"] == "EV Conditional").sum()
)

print(
    "Diesel Preferred:",
    (final_output["Final_EV_Decision"] == "Diesel Preferred").sum()
)

print(
    "High Priority:",
    (final_output["Transition_Priority"] == "High Priority").sum()
)

# ------------------------------------------------------------
# 11. TOP 10 TRANSITION PRIORITIES
# ------------------------------------------------------------

print("\n\nTOP 10 EV TRANSITION PRIORITIES")
print("=" * 100)

print(
    final_output[
        [
            "Final_EV_Rank",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Predicted_EV_Suitability",
            "Final_EV_Decision",
            "Transition_Priority",
            "Total_OPEX_Saving_Crore",
            "Potential_CO2_Reduction_Tonnes",
            "Total_EV_Energy_GWh"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

FINAL INTEGRATED EV TRANSITION ANALYSIS
 Final_EV_Rank  Depot ID      Depot Name           District Terrain_Class  Terrain_Score  Predicted_EV_Suitability  Historical_Transition_Score Final_EV_Decision Transition_Priority  Avg_KM_per_Bus  Avg_Passengers_per_Bus  Avg_Daily_KM_per_Bus  Economic_Benefit_Score  Operational_Utilization_Score  Route_Feasibility_Score  Total_KM  Total_Diesel_Litres  Total_CO2_Tonnes  Potential_CO2_Reduction_Tonnes  Total_EV_Energy_GWh  Total_OPEX_Saving_INR  Total_OPEX_Saving_Crore
             1 KSRTC-046     MAVELIKKARA          Alappuzha          Flat            1.0                  0.615878                     0.697494    EV Conditional       High Priority     6880.462290             9984.126747            226.228964                0.531246                       0.616651                      1.0  19536568         4.787788e+06      12831.271227                    12831.271227               24.423              468877632                46.887763
            

In [19]:
# ============================================================
# CELL 18 — FINAL IMPACT BASELINE: 2025 COMPLETE YEAR
# ============================================================

print("2025 COMPLETE-YEAR DEPOT IMPACT ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# 1. FILTER 2025 ONLY
# ------------------------------------------------------------

impact_2025 = monthly_df[
    monthly_df["Year"] == 2025
].copy()

print(
    "2025 records:",
    len(impact_2025)
)

print(
    "Expected records:",
    92 * 12
)

# ------------------------------------------------------------
# 2. AGGREGATE 2025 BY DEPOT
# ------------------------------------------------------------

impact_2025_depot = (
    impact_2025
    .groupby(
        [
            "Depot ID",
            "Depot Name",
            "District"
        ]
    )
    .agg(
        Baseline_2025_KM=(
            "Effective KM",
            "sum"
        ),
        Baseline_2025_Diesel_Litres=(
            "Estimated Diesel Litres",
            "sum"
        ),
        Baseline_2025_CO2_Tonnes=(
            "Estimated CO2 (Tonnes)",
            "sum"
        ),
        Baseline_2025_OPEX_Saving_INR=(
            "Potential EV OPEX Saving (INR)",
            "sum"
        ),
        Baseline_2025_EV_Energy_MWh=(
            "Estimated EV Energy (MWh)",
            "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. CONVERT UNITS
# ------------------------------------------------------------

impact_2025_depot["Baseline_2025_EV_Energy_GWh"] = (
    impact_2025_depot["Baseline_2025_EV_Energy_MWh"]
    / 1000
)

impact_2025_depot["Baseline_2025_OPEX_Saving_Crore"] = (
    impact_2025_depot["Baseline_2025_OPEX_Saving_INR"]
    / 1e7
)

# Potential avoided tailpipe CO2 under full diesel-to-EV
# transition scenario
impact_2025_depot["Baseline_2025_CO2_Reduction_Tonnes"] = (
    impact_2025_depot["Baseline_2025_CO2_Tonnes"]
)

# ------------------------------------------------------------
# 4. MERGE WITH FINAL ML RANKING
# ------------------------------------------------------------

final_2025_output = final_depots.merge(
    impact_2025_depot,
    on=[
        "Depot ID",
        "Depot Name",
        "District"
    ],
    how="left"
)

# ------------------------------------------------------------
# 5. VERIFY MERGE
# ------------------------------------------------------------

print("\nMERGE VALIDATION")
print("-" * 100)

print(
    "Depots in final output:",
    final_2025_output["Depot ID"].nunique()
)

print(
    "Missing 2025 impact rows:",
    final_2025_output["Baseline_2025_KM"].isna().sum()
)

# ------------------------------------------------------------
# 6. DISPLAY TOP 20
# ------------------------------------------------------------

print("\n\nTOP 20 DEPOTS — 2026 ML SUITABILITY + 2025 IMPACT")
print("=" * 120)

print(
    final_2025_output[
        [
            "Final_EV_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Predicted_EV_Suitability",
            "Final_EV_Decision",
            "Baseline_2025_KM",
            "Baseline_2025_OPEX_Saving_Crore",
            "Baseline_2025_CO2_Reduction_Tonnes",
            "Baseline_2025_EV_Energy_GWh"
        ]
    ]
    .sort_values(
        "Predicted_EV_Suitability",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. OVERALL 2025 IMPACT
# ------------------------------------------------------------

print("\n\n2025 OVERALL IMPACT")
print("=" * 100)

print(
    f"Total KM: "
    f"{final_2025_output['Baseline_2025_KM'].sum():,.0f}"
)

print(
    f"Potential OPEX saving: "
    f"₹{final_2025_output['Baseline_2025_OPEX_Saving_Crore'].sum():,.2f} crore"
)

print(
    f"Potential CO2 reduction: "
    f"{final_2025_output['Baseline_2025_CO2_Reduction_Tonnes'].sum():,.2f} tonnes"
)

print(
    f"EV energy requirement: "
    f"{final_2025_output['Baseline_2025_EV_Energy_GWh'].sum():,.2f} GWh"
)

# ------------------------------------------------------------
# 8. CHECK 2025 COVERAGE
# ------------------------------------------------------------

print("\n\n2025 DEPOT COVERAGE")
print("=" * 100)

print(
    "Depots with 2025 data:",
    impact_2025_depot["Depot ID"].nunique()
)

print("Expected depots: 92")

2025 COMPLETE-YEAR DEPOT IMPACT ANALYSIS
2025 records: 1104
Expected records: 1104

MERGE VALIDATION
----------------------------------------------------------------------------------------------------
Depots in final output: 92
Missing 2025 impact rows: 0


TOP 20 DEPOTS — 2026 ML SUITABILITY + 2025 IMPACT
 Final_EV_Rank  Depot ID      Depot Name           District Terrain_Class  Predicted_EV_Suitability Final_EV_Decision  Baseline_2025_KM  Baseline_2025_OPEX_Saving_Crore  Baseline_2025_CO2_Reduction_Tonnes  Baseline_2025_EV_Energy_GWh
             1 KSRTC-046     MAVELIKKARA          Alappuzha          Flat                  0.615878    EV Conditional           3944910                         9.467784                         2590.946894                        4.932
             2 KSRTC-010       CHENGANUR          Alappuzha          Flat                  0.615554    EV Conditional           3954520                         9.490848                         2597.258571                   

In [20]:
# ============================================================
# CELL 19 — FINAL TRANSITION PRIORITY SCORE
# ============================================================

print("FINAL EV TRANSITION PRIORITY ANALYSIS")
print("=" * 100)

priority_df = final_2025_output.copy()

# ------------------------------------------------------------
# 1. NORMALIZE ECONOMIC BENEFIT
# ------------------------------------------------------------

priority_df["Economic_Impact_Normalized"] = (
    priority_df["Baseline_2025_OPEX_Saving_Crore"]
    /
    priority_df["Baseline_2025_OPEX_Saving_Crore"].max()
)

# ------------------------------------------------------------
# 2. NORMALIZE CO2 BENEFIT
# ------------------------------------------------------------

priority_df["CO2_Impact_Normalized"] = (
    priority_df["Baseline_2025_CO2_Reduction_Tonnes"]
    /
    priority_df["Baseline_2025_CO2_Reduction_Tonnes"].max()
)

# ------------------------------------------------------------
# 3. USE EXISTING TERRAIN FEASIBILITY
# ------------------------------------------------------------

priority_df["Route_Feasibility_Normalized"] = (
    priority_df["Terrain_Score"]
)

# ------------------------------------------------------------
# 4. USE ML SUITABILITY
# ------------------------------------------------------------

priority_df["ML_Suitability_Normalized"] = (
    priority_df["Predicted_EV_Suitability"]
)

# ------------------------------------------------------------
# 5. FINAL TRANSITION PRIORITY SCORE
# ------------------------------------------------------------
#
# 40%  ML EV suitability
# 30%  Economic benefit
# 20%  CO2 reduction
# 10%  Route feasibility
#
# This is a PRIORITIZATION score.
# It is separate from the EV suitability target.

priority_df["Transition_Priority_Score"] = (
    0.40 * priority_df["ML_Suitability_Normalized"]
    +
    0.30 * priority_df["Economic_Impact_Normalized"]
    +
    0.20 * priority_df["CO2_Impact_Normalized"]
    +
    0.10 * priority_df["Route_Feasibility_Normalized"]
)

# ------------------------------------------------------------
# 6. RANK DEPOTS
# ------------------------------------------------------------

priority_df = (
    priority_df
    .sort_values(
        "Transition_Priority_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

priority_df["Transition_Priority_Rank"] = (
    priority_df.index + 1
)

# ------------------------------------------------------------
# 7. PRIORITY CATEGORY
# ------------------------------------------------------------

def priority_level(rank):

    if rank <= 10:
        return "High Priority"

    elif rank <= 30:
        return "Medium Priority"

    else:
        return "Lower Priority"


priority_df["Transition_Priority_Level"] = (
    priority_df["Transition_Priority_Rank"]
    .apply(priority_level)
)

# ------------------------------------------------------------
# 8. DISPLAY TOP 20
# ------------------------------------------------------------

print("\nTOP 20 DEPOTS FOR EV TRANSITION")
print("=" * 120)

print(
    priority_df[
        [
            "Transition_Priority_Rank",
            "Depot ID",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Predicted_EV_Suitability",
            "Transition_Priority_Score",
            "Final_EV_Decision",
            "Baseline_2025_OPEX_Saving_Crore",
            "Baseline_2025_CO2_Reduction_Tonnes",
            "Baseline_2025_EV_Energy_GWh"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. FINAL PRIORITY DISTRIBUTION
# ------------------------------------------------------------

print("\n\nTRANSITION PRIORITY DISTRIBUTION")
print("=" * 80)

print(
    priority_df[
        "Transition_Priority_Level"
    ].value_counts()
)

# ------------------------------------------------------------
# 10. EV DECISION × PRIORITY
# ------------------------------------------------------------

print("\n\nEV DECISION × TRANSITION PRIORITY")
print("=" * 80)

print(
    pd.crosstab(
        priority_df["Final_EV_Decision"],
        priority_df["Transition_Priority_Level"]
    )
)

# ------------------------------------------------------------
# 11. TOP 10 FINAL CANDIDATES
# ------------------------------------------------------------

print("\n\nTOP 10 FINAL EV TRANSITION CANDIDATES")
print("=" * 120)

print(
    priority_df[
        [
            "Transition_Priority_Rank",
            "Depot Name",
            "District",
            "Terrain_Class",
            "Predicted_EV_Suitability",
            "Transition_Priority_Score",
            "Final_EV_Decision",
            "Baseline_2025_OPEX_Saving_Crore",
            "Baseline_2025_CO2_Reduction_Tonnes",
            "Baseline_2025_EV_Energy_GWh"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

FINAL EV TRANSITION PRIORITY ANALYSIS

TOP 20 DEPOTS FOR EV TRANSITION
 Transition_Priority_Rank  Depot ID           Depot Name           District Terrain_Class  Predicted_EV_Suitability  Transition_Priority_Score Final_EV_Decision  Baseline_2025_OPEX_Saving_Crore  Baseline_2025_CO2_Reduction_Tonnes  Baseline_2025_EV_Energy_GWh
                        1 KSRTC-024               KANNUR             Kannur  Flat/Rolling                  0.585713                   0.824285    EV Conditional                        18.810317                         5147.617635                        9.797
                        2 KSRTC-032               KOLLAM             Kollam  Flat/Rolling                  0.585713                   0.800518    EV Conditional                        17.916197                         4902.933405                        9.330
                        3 KSRTC-022            KANGANGAD          Kasaragod  Flat/Rolling                  0.585726                   0.725193    EV Con